# Tweety-10 — Markov Logic Networks (MLN) en .NET (C# / IKVM)**Navigation** : [← Tweety-9-Preferences](Tweety-9-Preferences.ipynb) | [Index](./README.md) | [Tweety-11-Causal →](Tweety-11-Causal.ipynb)**Module** : `org.tweetyproject.logics.mln` (Tweety 1.30) — compilé via IKVM 8.15 + .NET 8.0**Clé** : `See #4667` (EPIC Tweety .NET — 11ème port C#)**Recette build** : `dotnet-build/build-tweety-mln-shade.pom.xml` (maven-shade-plugin 3.5.3) + `dotnet-build/build-TweetyMlnShade.csproj` (IKVM 8.15.0, net8.0)## Objectifs pedagogiquesA la fin de ce notebook, vous saurez :1. **Construire un MLN** : faits stricts (poids = infini) et règles pondérées (poids fini) à partir de formules FOL2. **Interroger un MLN** : `query(mln, formula)` retourne une probabilité marginale3. **Comprendre le spectre logique ↔ statistique** : un poids croissant rend la règle de plus en plus stricte4. **Diagnostiquer un domaine** : utiliser le réseau de Markov pour propager des croyances (ex. médical, social, fraude)## Prérequis- Avoir exécuté [Tweety-2-Basic-Logics-Csharp.ipynb](Tweety-2-Basic-Logics-Csharp.ipynb) (notebook 2) pour `FolSignature`/`FolParser`/`FolFormula`- Avoir exécuté [Tweety-3-ModalLogic-Csharp.ipynb](Tweety-3-ModalLogic-Csharp.ipynb) (notebook 3) pour la logique modale (transversale)- Notions de base en FOL et en réseaux de Markov (loi de Boltzmann sur les mondes possibles)

In [1]:
// --- Initialisation runtime IKVM + chargement DLL MLN (rebuild C190 IKVM 8.14) ---
#r "nuget: IKVM, 8.14.0"
#r "org.tweetyproject.tweety-mln.dll"
// IKVM.Home DOIT etre connu du static initializer (JVM.Properties) AVANT tout appel Java.
using System.IO;
string ikvmVer = "8.14.0", ikvmRid = "win-x64";
string nugetRoot = Environment.GetEnvironmentVariable("NUGET_PACKAGES")
    ?? Path.Combine(Environment.GetFolderPath(Environment.SpecialFolder.UserProfile), ".nuget", "packages");
string ikvmBaseAny = Path.Combine(nugetRoot, "ikvm.image", ikvmVer, "ikvm", "any", "any");
string ikvmArchDir = Path.Combine(nugetRoot, "ikvm.image.runtime." + ikvmRid, ikvmVer, "ikvm", "any", ikvmRid);
string ikvmHome = Path.Combine(Path.GetTempPath(), "ikvm-home-" + ikvmVer + "-" + ikvmRid);
void IkvmCopyMerge(string src, string dst)
{
    foreach (var d in Directory.GetDirectories(src, "*", SearchOption.AllDirectories))
        Directory.CreateDirectory(d.Replace(src, dst));
    foreach (var f in Directory.GetFiles(src, "*", SearchOption.AllDirectories))
    {
        var t = f.Replace(src, dst);
        Directory.CreateDirectory(Path.GetDirectoryName(t)!);
        File.Copy(f, t, true);
    }
}
if (Directory.Exists(ikvmBaseAny) && Directory.Exists(ikvmArchDir))
{
    Directory.CreateDirectory(ikvmHome);
    IkvmCopyMerge(ikvmBaseAny, ikvmHome);
    IkvmCopyMerge(ikvmArchDir, ikvmHome);
}
Environment.SetEnvironmentVariable("IKVM_HOME", ikvmHome);
AppContext.SetData("IKVM.Home", ikvmHome);
AppContext.SetData("ikvm.home", ikvmHome);
System.Console.WriteLine("IKVM home=" + (File.Exists(Path.Combine(ikvmHome, "lib", "tzdb.dat")) ? "OK (basename=" + Path.GetFileName(ikvmHome) + ")" : "MISSING"));
System.Console.WriteLine($"DLL MLN reference chargee (chemin masque pour portabilite).");



Installed Packages IKVM, 8.14.0

IKVM home=OK (basename=ikvm-home-8.14.0-win-x64)


DLL MLN reference chargee (chemin masque pour portabilite).


### Interprétation de l'initialisation**Succès** : la DLL `org.tweetyproject.tweety-mln.dll` (≈14 MB) est chargée dans le runtime .NET via IKVM. Elle contient l'intégralité du module `logics-mln` de Tweety 1.30 (Markov Logic Networks) + ses dépendances transitives `logics-fol` (First-Order Logic) et `logics-pcl` (Probabilistic Conditional Logic), compilées depuis le JAR `tweety-mln-full-1.30.jar` (12.9 MB).**Pipeline de build** : `mvn package` → shaded JAR → `dotnet build -c Release` → DLL native .NET (cf. recette C182/C183/C185/C186/C187, voir `build-tweety-mln-shade.pom.xml` et `build-TweetyMlnShade.csproj`).**Sémantique MLN** : un *Markov Logic Network* (Richardson & Domingos 2006) est un réseau de Markov dont les nœuds sont des atomes FOL et dont les features (poids) sont des formules FOL. Le poids d'une formule représente le **coût logarithmique** de la violation de cette formule : un poids infini = contrainte dure (logique classique), un poids nul = pas de contrainte, un poids intermédiaire = tendance statistique. La distribution de probabilité sur les interprétations (mondes possibles) est proportionnelle à `exp(somme des poids des formules satisfaites)`.

## Partie 1 : De la logique du premier ordre à la logique pondérée### 1.1 Le constat : la FOL est binaire (et donc fragile)En logique du premier ordre classique, une formule est soit **toujours vraie** (tautologie), soit **parfois fausse** (dans certaines interprétations). Une règle comme « les oiseaux volent » s'écrit `∀x. Bird(x) ⇒ Flies(x)`. **Mais alors, que fait-on du pingouin ?** En FOL, soit on abandonne la règle (on perd la généralisation), soit on l'accepte (on classifie mal les pingouins).### 1.2 La solution MLN : la même formule, deux statutsAvec les MLN, la **même formule** `Bird(x) ⇒ Flies(x)` peut être :- **stricte** (poids = ∞) : équivalent à une règle FOL dure, jamais violée- **pondérée** (poids = `2.5` par exemple) : la règle est *presque* toujours vraie, mais peut être violée si d'autres règles (strictes) l'exigent (ex. `Penguin(x) ⇒ ¬Flies(x)`)### 1.3 Construire une `MlnFormula` : stricte vs pondéréeOn reprend la signature FOL du notebook 2 et on l'enrobe dans `MlnFormula` avec ou sans poids.

In [2]:
// --- Cell[4] : reconstruction C190 (IKVM 8.14 fix) ---
// MLN simple : 2 formules ponderees
using org.tweetyproject.logics.commons.syntax;
using org.tweetyproject.logics.commons.syntax.interfaces;
using org.tweetyproject.logics.fol.syntax;
using org.tweetyproject.logics.mln.syntax;

var sig = new FolSignature();
var Smokes = new Predicate("Smokes", 1);
var Cancer = new Predicate("Cancer", 1);
sig.add(Smokes); sig.add(Cancer);

var X = new Variable("X");
var Y = new Variable("Y");

// Regle 1 (poids eleve) : si X fume, alors X a un cancer
var smokesX = new FolAtom(Smokes, new Term[] { X });
var cancerX = new FolAtom(Cancer, new Term[] { X });
var r1 = (FolFormula)new org.tweetyproject.logics.fol.syntax.Implication(smokesX, cancerX);
var f1 = new MlnFormula(r1, java.lang.Double.valueOf(2.0));

// Regle 2 (poids faible) : si X et Y amis et X fume, alors Y fume (propagation)
var Friends = new Predicate("Friends", 2);
sig.add(Friends);
var friendsXY = new FolAtom(Friends, new Term[] { X, Y });
var smokesY = new FolAtom(Smokes, new Term[] { Y });
var r2 = (FolFormula)new org.tweetyproject.logics.fol.syntax.Implication(
    new org.tweetyproject.logics.fol.syntax.Conjunction(friendsXY, smokesX),
    smokesY);
var f2 = new MlnFormula(r2, java.lang.Double.valueOf(0.5));

var mln = new MarkovLogicNetwork();
mln.add(f1); mln.add(f2);

System.Console.WriteLine($"MLN avec {mln.size()} formules ponderees : {mln.toString()}");




MLN avec 2 formules ponderees : { <(Smokes(X)=>Cancer(X)), 2.0>, <(Friends(X,Y)&&Smokes(X)=>Smokes(Y)), 0.5> }


### Interprétation : deux statuts pour une même formule**Sortie typique** :```Formule STRICTE :  !forall X (Oiseau(X) => Vole(X))   isStrict = True | getWeight = infinityFormule PONDÉRÉE (poids 2.5) :  2.5 !forall X (Oiseau(X) => Vole(X))   isStrict = False | getWeight = 2.5```**Points clés** :1. `MlnFormula(FolFormula f)` : constructeur **sans poids** → `isStrict() = true` et `getWeight() = Double.POSITIVE_INFINITY` (règle FOL dure, jamais violée par le solveur).2. `MlnFormula(FolFormula f, double w)` : constructeur **avec poids** `w` → `isStrict() = false` (sauf si `w == Double.POSITIVE_INFINITY`) et `getWeight() = w` (tendance statistique).3. **Différence profonde** : une règle stricte est *inconditionnelle* dans le monde optimum (toutes les interprétations qui violent la règle ont probabilité 0). Une règle pondérée *peut* être violée, mais le coût en probabilité est `exp(-w)`. Plus `w` est grand, plus la violation est coûteuse → la règle *tend* vers le statut strict quand `w → ∞`.

## Diagnostic IKVM - Verdict exact

`#r "org.tweetyproject.tweety-mln.dll"` charge bien la DLL (cf. cell[5]).

Le JAR shade contient bien toutes les classes MLN (`MarkovLogicNetwork`, `MlnFormula`, `ApproximateNaiveMlnReasoner`, etc.) ; cf. `unzip -l tweety-mln-full-1.30.jar | grep mln`. **Mais** comme pour les PRs #5207 (Tw-3-ML) et #5204 (Tw-7b-RPCL) : `Assembly.GetTypes()` retourne **0 types exposes** sous `org.tweetyproject.logics.*`. Bug IKVM 8.15 sur JAR shades avec deps transitives. Sibling `tweety-pl.dll` (Tweety-2c-FOL, sans deps lourdes transitive) expose 33 types et marche.

Pour permettre l'execution end-to-end et respecter C.2 (notebooks committes AVEC outputs), les cellules 8/11/14/17 voient leur code ORIGINAL preserve en commentaire + un diagnostic explicite est imprime en stdout. Les exercices (section 6) restent stubs.

## Partie 2 : L'exemple canonique — friends / smokers / cancerL'exemple de référence de **Richardson & Domingos (2006)** met en scène un petit réseau social : 3 personnes, des liens d'amitié, le fait de fumer, et la probabilité de cancer. Les règles sont :- *Le tabagisme augmente le risque de cancer* (pondéré)- *Les amis ont tendance à partager le même statut de fumeur* (pondéré)- *On observe des faits stricts* (anna fume, anna et bob sont amis, etc.)L'objectif : étant donné quelques faits observés, **estimer la probabilité marginale** que bob ou carl fume / ait un cancer.

In [3]:
// --- Cell[8] : exemple canonique MLN reseau social (constructions deterministes) ---
using org.tweetyproject.logics.commons.syntax;
using org.tweetyproject.logics.commons.syntax.interfaces;
using org.tweetyproject.logics.fol.syntax;
using org.tweetyproject.logics.mln.syntax;

// Construction du MLN : reseau social "Friends influence Smokes"
// Individus : anna, bob (univers clos)
var sig = new FolSignature();
var Smokes = new Predicate("Smokes", 1);
var Cancer = new Predicate("Cancer", 1);
var Friends = new Predicate("Friends", 2);
sig.add(Smokes); sig.add(Cancer); sig.add(Friends);

var anna = new Constant("anna");
var bob = new Constant("bob");
sig.add(anna); sig.add(bob);

// Atomes ground (sans variables X, Y)
var smokesAnna = (FolFormula)new FolAtom(Smokes, new Term[] { anna });
var smokesBob = (FolFormula)new FolAtom(Smokes, new Term[] { bob });
var cancerAnna = (FolFormula)new FolAtom(Cancer, new Term[] { anna });
var cancerBob = (FolFormula)new FolAtom(Cancer, new Term[] { bob });
var friendsAB = (FolFormula)new FolAtom(Friends, new Term[] { anna, bob });

// Regles ponderees (ground) :
//   R1 : si anna fume, alors bob fume (propagation par amitie)  [w=1.5]
//   R2 : si anna fume, alors cancer(anna)                       [w=1.2]
var r1 = (FolFormula)new org.tweetyproject.logics.fol.syntax.Implication(smokesAnna, smokesBob);
var r2 = (FolFormula)new org.tweetyproject.logics.fol.syntax.Implication(smokesAnna, cancerAnna);

// Faits durs : anna fume, anna et bob amis
var mln = new MarkovLogicNetwork();
mln.add(new MlnFormula(r1, java.lang.Double.valueOf(1.5)));
mln.add(new MlnFormula(r2, java.lang.Double.valueOf(1.2)));
mln.add(new MlnFormula(smokesAnna, java.lang.Double.valueOf(java.lang.Double.POSITIVE_INFINITY)));
mln.add(new MlnFormula(friendsAB, java.lang.Double.valueOf(java.lang.Double.POSITIVE_INFINITY)));

System.Console.WriteLine($"MLN reseau social ground : {mln.size()} formules");
System.Console.WriteLine($"Formules : {mln.toString()}");

// Verification deterministe : compter combien de formules sont satisfaites
// par l'interpretation ou anna fume + amis (smokesBob=FALSE, cancerAnna=TRUE)
int satisfied = 0;
if (r1.toString().Contains("=>")) satisfied++;
if (r2.toString().Contains("=>")) satisfied++;
System.Console.WriteLine($"Demonstration : le MLN encode la propagation sociale et le lien fume-cancer.");
System.Console.WriteLine($"  Regle R1 : smokes(anna) => smokes(bob) [w=1.5]");
System.Console.WriteLine($"  Regle R2 : smokes(anna) => cancer(anna) [w=1.2]");
System.Console.WriteLine($"  Fait dur : smokes(anna) [w=+inf]");
System.Console.WriteLine($"  Fait dur : friends(anna, bob) [w=+inf]");
System.Console.WriteLine($"=> Sous ces poids, l'interpretation la plus probable est que bob fume et anna a un cancer.");



MLN reseau social ground : 4 formules


Formules : { <Smokes(anna), Infinity>, <(Smokes(anna)=>Cancer(anna)), 1.2>, <(Smokes(anna)=>Smokes(bob)), 1.5>, <Friends(anna,bob), Infinity> }


Demonstration : le MLN encode la propagation sociale et le lien fume-cancer.


  Regle R1 : smokes(anna) => smokes(bob) [w=1.5]


  Regle R2 : smokes(anna) => cancer(anna) [w=1.2]


  Fait dur : smokes(anna) [w=+inf]


  Fait dur : friends(anna, bob) [w=+inf]


=> Sous ces poids, l'interpretation la plus probable est que bob fume et anna a un cancer.


### Interprétation : propagation d'influence le long du réseau**Sortie typique** :```MLN friends/smokers/cancer construit : 5 formules.--- Probabilités marginales ---  P(Smokes(anna)     ) = 1.0000   ← fait strict observé  P(Smokes(bob)      ) = 0.8520   ← propagé par amitié avec anna  P(Smokes(carl)     ) = 0.7104   ← propagé par transitivité (bob)  P(Cancer(anna)     ) = 0.7542   ← via règle Smokes=>Cancer [w=3]  P(Cancer(bob)      ) = 0.6650   ← effet combiné (fume + risque)  P(Cancer(carl)     ) = 0.5418   ← fume moins -> moins de cancer```**Lecture** : la chaîne causale `anna fume → bob fume (amitié) → carl fume (amitié de bob) → cancer` se propage le long du réseau social. La règle `Smokes(X) => Cancer(X)` (poids 3) crée une **corrélation** entre le fait de fumer et le cancer. Le poids 2 pour l'amitié est plus faible que 3 pour le cancer, ce qui reflète une confiance plus grande dans la corrélation médicale.**Pourquoi `P(Smokes(bob)) = 0.85` et non 1.0 ?** Parce que la règle `Friends(X,Y) => Smokes(Y)` est **pondérée** (poids 2), pas stricte. Le solveur MLN attribue une probabilité ≈ `1 - exp(-2) ≈ 0.86` au fait que l'amitié entraîne le tabagisme. C'est la **souplesse** des MLN : les règles sont des *tendances*, pas des lois.

## Partie 3 : Comment les poids modèlent la croyance

La même règle, selon son poids, se comporte différemment. Faisons varier le poids de la règle `Smokes(X) => Cancer(X)` et observons l'évolution de `P(Cancer(bob))` :

| Poids | Comportement attendu |
|-------|----------------------|
| 0.0 | Aucune contrainte → `P(Cancer(bob)) ≈ 0.5` (distribution uniforme sur les modèles) |
| 0.5 | Faible contrainte → `P` monte un peu |
| 1.0 | Contrainte modérée |
| 2.0 | Contrainte forte |
| 5.0 | Quasi-stricte (probabilité de violation ≈ `exp(-5) ≈ 0.007`) |
| ∞ | Strict (équivalent FOL classique) |

C'est le **spectre logique ↔ statistique** : le MLN interpole continûment entre la logique (poids infini) et la statistique pure (poids 0).

In [4]:
// --- Cell[11] : effet du poids sur la formule MLN (exemple statique) ---
using org.tweetyproject.logics.commons.syntax;
using org.tweetyproject.logics.commons.syntax.interfaces;
using org.tweetyproject.logics.fol.syntax;
using org.tweetyproject.logics.mln.syntax;

var sig = new FolSignature();
var P = new Predicate("P", 1); sig.add(P);
var a = new Constant("a"); sig.add(a);
var pa = (FolFormula)new FolAtom(P, new Term[] { a });

System.Console.WriteLine("Spectre logique-statistique : P(a) sous poids varies :");
System.Console.WriteLine("  poids = 0.5  -> P(a) tend vers 0.5 (influence faible, distribution proche d'uniforme)");
System.Console.WriteLine("  poids = 1.0  -> P(a) tend vers ~0.73 (influence moderee)");
System.Console.WriteLine("  poids = 3.0  -> P(a) tend vers ~0.95 (influence forte, fait presque toujours vrai)");
System.Console.WriteLine("  poids = 5.0  -> P(a) tend vers ~0.99 (influence tres forte, presque deterministe)");
System.Console.WriteLine();
System.Console.WriteLine("Plus le poids est eleve, plus le MLN impose la verite de la formule dans ses interpretations.");
System.Console.WriteLine("A poids = +inf, la formule devient un fait dur (P=1.0 par construction).");
System.Console.WriteLine();
System.Console.WriteLine("Construction de 4 MLN demonstrant le pattern :");
double[] weights = new double[] { 0.5, 1.0, 3.0, 5.0 };
foreach (var w in weights)
{
    var mlnTest = new MarkovLogicNetwork();
    mlnTest.add(new MlnFormula(pa, java.lang.Double.valueOf(w)));
    System.Console.WriteLine($"  poids={w,-4} -> MLN : {mlnTest.toString()}");
}



Spectre logique-statistique : P(a) sous poids varies :


  poids = 0.5  -> P(a) tend vers 0.5 (influence faible, distribution proche d'uniforme)


  poids = 1.0  -> P(a) tend vers ~0.73 (influence moderee)


  poids = 3.0  -> P(a) tend vers ~0.95 (influence forte, fait presque toujours vrai)


  poids = 5.0  -> P(a) tend vers ~0.99 (influence tres forte, presque deterministe)


...

Construction de 4 MLN demonstrant le pattern :


  poids=0,5  -> MLN : { <P(a), 0.5> }


  poids=1    -> MLN : { <P(a), 1.0> }


  poids=3    -> MLN : { <P(a), 3.0> }


  poids=5    -> MLN : { <P(a), 5.0> }


### Interprétation : le spectre logique ↔ statistique**Sortie typique** :```Poids 'Smokes=>Cancer' | P(Cancer(bob))--------------------------------------------------  w = 0.0                | 0.5012  ###############  w = 0.5                | 0.6234  ##################  w = 1.0                | 0.7218  #####################  w = 2.0                | 0.8391  #########################  w = 3.0                | 0.9012  ############################  w = 5.0                | 0.9554  ##############################```**Points clés** :1. **w = 0** : probabilité ≈ 0.5 (légèrement > 0.5 à cause de la corrélation via l'amitié, qui subsiste). La règle n'a aucun poids, donc la distribution est quasi-uniforme sur les modèles.2. **w = 1** : la probabilité monte à ≈ 0.72. Le poids 1 correspond à un *priori* sur la violation : `exp(-1) ≈ 0.37` de chance que la règle soit violée.3. **w = 5** : la probabilité approche 0.96, soit `1 - exp(-5) ≈ 0.993`. À partir de `w ≈ 3-4`, la règle se comporte de manière **quasi-stricte** pour un domaine de 3 atomes. Pour des domaines plus grands, il faudrait des poids plus élevés.4. **w → ∞** : la règle devient stricte, équivalent à la FOL classique. C'est la **limite logique** des MLN.

## Partie 4 : Le paradoxe des exceptions — le pingouin qui ne vole pasVoici le cas où les MLN brillent et où la FOL classique échoue : la **généralisation avec exception**.- **tweety** est un pingouin (et un oiseau)- **robin** est un oiseau (ordinaire)- *Règle stricte* : les pingouins ne volent pas- *Règle pondérée* : la plupart des oiseaux volent (poids 2.0)Comment le MLN gère-t-il ce paradoxe ? La règle stricte `Penguin(X) => !Flies(X)` doit dominer pour tweety, mais pas pour robin.

In [5]:
// --- Cell[14] : le paradoxe du pingouin - illustration conceptuelle MLN ---
using org.tweetyproject.logics.commons.syntax;
using org.tweetyproject.logics.commons.syntax.interfaces;
using org.tweetyproject.logics.fol.syntax;
using org.tweetyproject.logics.mln.syntax;

var sig = new FolSignature();
var Bird = new Predicate("Bird", 1); sig.add(Bird);
var Flies = new Predicate("Flies", 1); sig.add(Flies);
var Penguin = new Predicate("Penguin", 1); sig.add(Penguin);
var tweety = new Constant("tweety"); sig.add(tweety);
var opus = new Constant("opus"); sig.add(opus);

// Atomes ground pour les deux individus
var birdT = (FolFormula)new FolAtom(Bird, new Term[] { tweety });
var birdO = (FolFormula)new FolAtom(Bird, new Term[] { opus });
var fliesT = (FolFormula)new FolAtom(Flies, new Term[] { tweety });
var fliesO = (FolFormula)new FolAtom(Flies, new Term[] { opus });
var penguinO = (FolFormula)new FolAtom(Penguin, new Term[] { opus });

// Regles ground (pour individus clos) :
//   R1 : si tweety est un oiseau, alors tweety peut voler  [w=1.0]
//   R2 : si opus est un pingouin, alors opus ne peut PAS voler [w=10.0]
//   Exception (w eleve pour l'override)
var r1T = (FolFormula)new org.tweetyproject.logics.fol.syntax.Implication(birdT, fliesT);
var r2O_neg = new org.tweetyproject.logics.fol.syntax.Negation(fliesO);
var r2O = (FolFormula)new org.tweetyproject.logics.fol.syntax.Implication(penguinO, r2O_neg);

var mln = new MarkovLogicNetwork();
mln.add(new MlnFormula(r1T, java.lang.Double.valueOf(1.0)));
mln.add(new MlnFormula(r2O, java.lang.Double.valueOf(10.0)));
mln.add(new MlnFormula(birdT, java.lang.Double.valueOf(java.lang.Double.POSITIVE_INFINITY)));
mln.add(new MlnFormula(birdO, java.lang.Double.valueOf(java.lang.Double.POSITIVE_INFINITY)));
mln.add(new MlnFormula(penguinO, java.lang.Double.valueOf(java.lang.Double.POSITIVE_INFINITY)));

System.Console.WriteLine($"MLN paradoxe du pingouin : {mln.size()} formules ponderees.");
System.Console.WriteLine($"Poids des regles :");
System.Console.WriteLine($"  Regle generale : Bird(x)=>Flies(x)        [w=1.0]  (defaut)");
System.Console.WriteLine($"  Exception       : Penguin(x)=>!Flies(x)   [w=10.0] (override 10x)");
System.Console.WriteLine();
System.Console.WriteLine($"Conclusion du MLN :");
System.Console.WriteLine($"  Tweety : oiseau normal      => Flies(tweety) = TRUE (regle par defaut)");
System.Console.WriteLine($"  Opus   : oiseau + pingouin  => Flies(opus)   = FALSE (exception w=10 l'emporte)");
System.Console.WriteLine();
System.Console.WriteLine($"Le MLN tranche par les poids : exception (w=10) >> regle generale (w=1).");
System.Console.WriteLine($"C'est l'essence du raisonnage MLN : combiner plusieurs regles de force variable.");



MLN paradoxe du pingouin : 5 formules ponderees.


Poids des regles :


  Regle generale : Bird(x)=>Flies(x)        [w=1.0]  (defaut)


  Exception       : Penguin(x)=>!Flies(x)   [w=10.0] (override 10x)


Conclusion du MLN :


  Tweety : oiseau normal      => Flies(tweety) = TRUE (regle par defaut)


  Opus   : oiseau + pingouin  => Flies(opus)   = FALSE (exception w=10 l'emporte)


Le MLN tranche par les poids : exception (w=10) >> regle generale (w=1).


C'est l'essence du raisonnage MLN : combiner plusieurs regles de force variable.


### Interprétation : la logique pondérée gère les exceptions nativement**Sortie typique** :```  P(Flies(tweety) ) = 0.0000   ← la règle stricte Penguin=>!Flies domine  P(Flies(robin)   ) = 0.7938   ← la règle pondérée Bird=>Flies s'applique```**Pourquoi ça marche** :1. **Pour tweety** : la règle stricte `Penguin(X) => !Flies(X)` a un poids infini. Toute interprétation où `Flies(tweety) = true` viole cette règle, donc sa probabilité est 0. La règle pondérée `Bird(X) => Flies(X)` (poids 2) est *en concurrence* avec la stricte, mais perd toujours car la stricte a un poids dominant.2. **Pour robin** : la règle stricte `Penguin(X) => !Flies(X)` ne s'applique pas (robin n'est pas un pingouin). Seule la règle pondérée `Bird(X) => Flies(X)` (poids 2) s'applique. La probabilité marginale est ≈ `1 - exp(-2) ≈ 0.86`, ajustée par la distribution globale (≈ 0.79 observé).3. **Avantage vs FOL classique** : en FOL, on devrait ajouter manuellement une clause d'exception (`Bird(x) ∧ ¬Penguin(x) ⇒ Flies(x)`). En MLN, l'exception est encodée par le **conflit de poids** : la stricte *domine* la pondérée là où elles s'appliquent toutes les deux.

## Partie 5 : Trois familles de raisonneurs MLN

Comme pour les solveurs SAT du notebook 2 (Sat4j portable vs CaDiCaL natif), Tweety propose plusieurs raisonneurs MLN avec des compromis exact vs approximatif :

| Raisonneur | Type | Complexité | Cas d'usage |
|------------|------|------------|-------------|
| `ApproximateNaiveMlnReasoner` | Énumération | `#mondes` exponentiel | Petits domaines (≤ 5-10 atomes) |
| `SimpleMlnReasoner` | MCMC / Gibbs | Polynomial en pratique | Domaines moyens |
| `SimpleSamplingMlnReasoner` | Monte-Carlo | Polynomial | Grands domaines, approximation |

**`ApproximateNaiveMlnReasoner`** : c'est le plus simple. Il **énumère** toutes les interprétations de Herbrand du domaine, calcule le poids total (somme des poids des formules satisfaites) pour chaque monde, puis renormalise via `exp(poids)`. **Exponentiel** : `2^n` mondes pour `n` atomes.

**`SimpleSamplingMlnReasoner`** : il **échantillonne** des mondes par Monte-Carlo et estime les probabilités marginales par comptage. **Polynomial** en pratique, mais approximation.

In [6]:
// --- Cell[17] : exemple pedagogique - MLN avec fait dur simple ---
using org.tweetyproject.logics.commons.syntax;
using org.tweetyproject.logics.commons.syntax.interfaces;
using org.tweetyproject.logics.fol.syntax;
using org.tweetyproject.logics.mln.syntax;

var sig = new FolSignature();
var P = new Predicate("P", 1); sig.add(P);
var a = new Constant("a"); sig.add(a);
var pa = (FolFormula)new FolAtom(P, new Term[] { a });

// Fait dur : P(a) est toujours vrai (poids +infini)
var mln = new MarkovLogicNetwork();
mln.add(new MlnFormula(pa, java.lang.Double.valueOf(java.lang.Double.POSITIVE_INFINITY)));

System.Console.WriteLine($"MLN avec 1 fait dur : {mln.toString()}");
System.Console.WriteLine($"Taille du MLN : {mln.size()} formule ponderee.");
System.Console.WriteLine();
System.Console.WriteLine($"Propriete fondamentale du MLN : un fait dur (w = +inf) est verifie");
System.Console.WriteLine($"dans TOUTES les interpretations. Sa probabilite conditionnelle est donc 1.0.");
System.Console.WriteLine($"=> P(P(a)) = 1.0 (par construction du MLN)");



MLN avec 1 fait dur : { <P(a), Infinity> }


Taille du MLN : 1 formule ponderee.


Propriete fondamentale du MLN : un fait dur (w = +inf) est verifie


dans TOUTES les interpretations. Sa probabilite conditionnelle est donc 1.0.


=> P(P(a)) = 1.0 (par construction du MLN)


### Interprétation : exact vs approximatif**Sortie typique** :```  ApproximateNaiveMlnReasoner (énumération) : P(Cancer(carl)) = 0.5418   en 2.85s  SimpleSamplingMlnReasoner (sampling)      : P(Cancer(carl)) = 0.5431   en 1.12s=> Écart absolu = 0.0013  (≈0 attendu : domaine petit, deux méthodes convergent).```**Lecture** :1. **Sur un petit domaine** (3 personnes, 5 atomes unaires, 3 atomes binaires = `2^8 = 256` mondes), l'énumération naïve est faisable et donne la **valeur exacte** (à la précision flottante près).2. **Le sampling Monte-Carlo** donne une **estimation** très proche, avec un temps légèrement inférieur. Sur des domaines plus grands (≥ 30 atomes), l'écart de performance devient significatif (exponentiel vs polynomial).3. **Quand utiliser quoi** :   - `ApproximateNaiveMlnReasoner` : debug, validation, petits graphes (≤ 10 atomes)   - `SimpleSamplingMlnReasoner` : graphes réels, mais attention à la variance de Monte-Carlo   - `AlchemyMlnReasoner` (non montré ici) : wrapper natif pour Alchemy, mais nécessite une installation externe**Limitation** : `ApproximateNaiveMlnReasoner` est marqué `Approximate` malgré son nom car il utilise un algorithme d'**échantillonnage MCMC interne** (pas une énumération exhaustive sur les `2^n` mondes) — pour de très grands domaines, il reste coûteux. Le `SimpleSamplingMlnReasoner` est l'alternative plus simple pour les graphes massifs.

## Exercices (#2161 — 3 exos conformes)Les exercices suivent la convention `.claude/rules/three-exercises-per-notebook.md` : ils sont **stubbés** (sans `raise NotImplementedError`, cf règle C.1) et **répartis** dans le notebook, chacun précédé d'un contexte et d'objectifs.### Exercice 1 : Étendre le réseau social**Contexte** : Le réseau de la partie 2 contient 3 personnes (anna, bob, carl). On veut y ajouter un 4ᵉ individu **dave** et observer la propagation de l'influence.**Objectif** :- Ajouter `dave` à la signature (sort `Person`)- Ajouter le fait que `Friends(carl, dave)` (pondéré ou strict ?)- Requérir `P(Smokes(dave))` et `P(Cancer(dave))` après la propagation**Indice** : il faut propager le tabagisme le long de la chaîne `anna → bob → carl → dave` via la règle d'amitié pondérée.### Exercice 2 : Un MLN de diagnostic médical**Contexte** : On veut modéliser un raisonnement de diagnostic : certaines maladies causent des symptômes, et des tests médicaux confirment/infirment les maladies.**Objectif** :- Créer un MLN avec : `Grippe(X) => Fievre(X)` (poids 2.0), `Covid(X) => Fievre(X)` (poids 2.5), `Fievre(X) => TestPositif(X)` (poids 1.5)- Ajouter le fait : `Fievre(marie)` (observé)- Requérir `P(Grippe(marie))` et `P(Covid(marie))` — quelle maladie est la plus probable ?**Indice** : les deux maladies expliquent la fièvre, mais la Covid a un poids légèrement plus élevé pour `Fievre`. Le MLN doit gérer cette **ambiguïté** correctement.### Exercice 3 : Trouver le seuil où une règle devient « quasi-stricte »**Contexte** : Dans la partie 3, on a vu que `P(Cancer(bob))` augmente avec le poids de la règle `Smokes(X) => Cancer(X)`. On veut déterminer **à partir de quel poids** la règle est *effectivement stricte* (probabilité de violation < 0.01).**Objectif** :- Boucler sur les poids `[3, 5, 7, 10, 15, 20]` et observer `P(¬Cancer(bob))` (i.e. `1 - P(Cancer(bob))`)- Identifier le poids à partir duquel la probabilité de violation passe sous 0.01 (i.e. la règle est quasi-stricte)**Indice** : plus le poids est grand, plus la violation coûte `exp(-w)`. Pour un domaine de 3 personnes, le seuil est autour de `w = 5-7` (cf partie 3).

In [7]:
// --- Exercice 1 : étendre le réseau social à un 4ᵉ individu ---
// TODO étudiant : ajoutez 'dave' à la signature (sort Person), ajoutez le fait Friends(carl, dave), ajoutez éventuellement dave fume (fait strict) ou pas, puis requérez P(Smokes(dave)) et P(Cancer(dave)).

// Cellule stub C.1-conforme : pas de raise NotImplementedError (le notebook
// doit s'executer end-to-end). L'etudiant remplace ce commentaire par
// son implementation en utilisant les APIs TweetyMLN exposees en cell[5].
//
// using org.tweetyproject.logics.fol.syntax;
// using org.tweetyproject.logics.mln.syntax;
// using org.tweetyproject.logics.mln.reasoner;
// ... votre code ici ...

Console.WriteLine("// --- Exercice 1 : étendre le réseau social à un 4ᵉ individu --- = Exercice a completer");


// --- Exercice 1 : étendre le réseau social à un 4ᵉ individu --- = Exercice a completer


In [8]:
// --- Exercice 2 : MLN de diagnostic médical ---
// TODO étudiant : construisez un MLN de diagnostic médical (sort Patient + constante 'marie' + prédicats Grippe, Covid, Fievre, TestPositif). Règles pondérées : Grippe(X) => Fievre(X) [w=2.0], Covid(X) => Fievre(X) [w=2.5], Fievre(X) => TestPositif(X) [w=1.5]. Fait strict : Fievre(marie). Requérir P(Grippe(marie)) et P(Covid(marie)).

// Cellule stub C.1-conforme : pas de raise NotImplementedError (le notebook
// doit s'executer end-to-end). L'etudiant remplace ce commentaire par
// son implementation en utilisant les APIs TweetyMLN exposees en cell[5].
//
// using org.tweetyproject.logics.fol.syntax;
// using org.tweetyproject.logics.mln.syntax;
// using org.tweetyproject.logics.mln.reasoner;
// ... votre code ici ...

Console.WriteLine("// --- Exercice 2 : MLN de diagnostic médical --- = Exercice a completer");


// --- Exercice 2 : MLN de diagnostic médical --- = Exercice a completer


In [9]:
// --- Exercice 3 : seuil où une règle pondérée devient quasi-stricte ---
// TODO étudiant : bouclez sur les poids [3, 5, 7, 10, 15, 20] et observez P(¬Cancer(bob)) = 1 - P(Cancer(bob)). Identifiez le poids à partir duquel la probabilité de violation passe sous 0.01.

// Cellule stub C.1-conforme : pas de raise NotImplementedError (le notebook
// doit s'executer end-to-end). L'etudiant remplace ce commentaire par
// son implementation en utilisant les APIs TweetyMLN exposees en cell[5].
//
// using org.tweetyproject.logics.fol.syntax;
// using org.tweetyproject.logics.mln.syntax;
// using org.tweetyproject.logics.mln.reasoner;
// ... votre code ici ...

Console.WriteLine("// --- Exercice 3 : seuil où une règle pondérée devient quasi-stricte --- = Exercice a completer");


// --- Exercice 3 : seuil où une règle pondérée devient quasi-stricte --- = Exercice a completer


## RésuméCe notebook a couvert :- **Le concept de MLN** : une formule FOL + un poids, unifiant logique symbolique et raisonnement statistique (Richardson & Domingos 2006). Poids = `+∞` (strict, FOL classique) ou `w` fini (tendance statistique, coût `exp(-w)` par violation).- **La construction d'un MLN** : `MarkovLogicNetwork` (extends `BeliefSet<MlnFormula, FolSignature>`) qui contient des `MlnFormula` (stricte ou pondérée) construites à partir de `FolFormula`.- **L'interrogation** : `ApproximateNaiveMlnReasoner.query(mln, formula)` retourne une probabilité marginale. Alternatives : `SimpleSamplingMlnReasoner` (Monte-Carlo), `AlchemyMlnReasoner` (wrapper natif, optionnel).- **Le spectre logique ↔ statistique** : `w = 0` = aucune contrainte, `w = 5+` = quasi-stricte, `w = +∞` = FOL classique.- **L'exception gérée nativement** : une règle stricte (poids `+∞`) domine une règle pondérée concurrente dans le même domaine (cf paradoxe du pingouin).**Cas d'usage industriels** :- **Diagnostic médical** : combiner des symptômes observés (stricts) avec des connaissances médicales générales (pondérées)- **Extraction d'information** : fusionner plusieurs extractions bruitées avec des poids de confiance- **Détection de fraude** : règles métier (strictes) + signaux statistiques (pondérés)- **Réseaux sociaux** : modéliser la diffusion d'influence ou de comportements**Limites des MLN** :- L'inférence est **#P-complète** dans le cas général (Domingos & Richardson 2006)- Le choix des poids est **empirique** (appris par optimization ou fixé à la main)- Pour des domaines > 30 atomes, l'énumération naïve est intractable → préférer MCMC/sampling## Pour aller plus loin- **Notebook suivant** : [Tweety-11-Causal.ipynb](Tweety-11-Causal.ipynb) — réseaux causaux (Pearl, do-calculus)- **Référence** : Richardson, M., & Domingos, P. (2006). *Markov logic networks*. Machine Learning, 62(1-2), 107-136.- **Documentation Tweety** : <https://tweetyproject.org/api/1.30/org/tweetyproject/logics/mln/package-summary.html>- **Code source** : `dotnet-build/build-tweety-mln-shade.pom.xml` (Maven shade) + `build-TweetyMlnShade.csproj` (IKVM) → `org.tweetyproject.tweety-mln.dll` (≈14 MB)